[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/04_surrogate_modeling.ipynb)

# 04 — Surrogate Modeling

**Purpose.** Train `f_sur: (x, tilt) -> R_hat` and decide whether it is accurate
enough to optimise against. PROJECT.md section 16, Phase 5.

The surrogate predicts the **radio map**, not the KPIs (Decision 6). `src/kpi/`
then derives the five KPIs from the prediction exactly as it does from a
ray-traced map, so acceptance is judged at two levels: map error in dB, and the
KPI error that map error produces.

**The question is not "is the surrogate good".** It is *"is it good enough that
an optimization result computed against it can be defended"*. Those are different
questions, and only the second one matters here — which is why section 9 is a
decision, not a summary.

**Inputs.** `data/processed/surrogate_dataset.parquet` from notebook 03.

**Outputs.** `models/surrogate.pkl`, frozen, plus a two-level error report.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook except the COLAB_PACKAGES line below, which names
# the extras this particular notebook needs. Forked the repository? Change these
# three values and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/external/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load D_sur and split on scenarios

Whole **scenarios**, never configurations and never grid cells (PROJECT.md
section 12.3, Decision 8).

Cells from one configuration share the same tilts and most of the same
propagation paths. Configurations from one scenario share the buildings, the
materials and the UE trajectories. Splitting inside either reports an error far
below the real one — and the surrogate then looks accurate right up to the point
an optimizer relies on it.

The scenario boundary is the stricter of the two, and it is the one that makes
the held-out error a *sim-to-reality* claim rather than an interpolation claim.

In [ ]:
from src.surrogate import dataset, features

d_sur = dataset.load(cfg)
train_df, val_df, test_df = dataset.split(d_sur, cfg)

for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(f"{name:6} {len(part):5} samples  {part['scenario_id'].nunique():3} scenarios")

overlap = set(train_df["scenario_id"]) & set(test_df["scenario_id"])
assert not overlap, f"scenario leak across the split: {overlap}"

## 3. Build the features and fit the transform

`build_state` assembles what does not vary with tilt — cell geometry, band
identity, UE density, and the scene descriptors that let the model generalise
across perturbed scenarios.

`fit` is the **only** place in this project that learns from data. It sees the
training partition only: a scaler fitted on the full `D_sur` has seen the
held-out scenarios, and the reported error is then optimistic for a reason that
is very hard to find later.

In [ ]:
from src.data.load import load_cell_config, load_processed
from src.data.ue_density import ue_density
from src.radio import cell_band

table = cell_band.build_table(load_cell_config(cfg), cfg)
rho = ue_density(load_processed(cfg, "train"), cfg)

state = features.build_state(table, rho, cfg)
transformer = features.fit(train_df, state, cfg)  # TRAIN ONLY

x_train = features.transform(dataset.tilt_matrix(train_df), state, transformer)
x_val = features.transform(dataset.tilt_matrix(val_df), state, transformer)
x_test = features.transform(dataset.tilt_matrix(test_df), state, transformer)
x_train.shape

## 4. Train

Instantiated from `configs/surrogate.yaml` through Hydra's `_target_`, so
swapping architectures is a config edit and an ablation is a sweep.

The target is the RSRP tensor in dBm, indexed `(cell_band, grid_cell)` in table
order. The loss is mean squared error over all locations, cells and bands
(PROJECT.md section 11.3)::

    L_MSE = 1 / (|G| |C| |B|) * sum_g sum_i sum_b (R_hat(i,b,g) - R(i,b,g))^2

This is a dense spatial prediction problem, not a five-output regression.

In [ ]:
from hydra.utils import instantiate

from src.utils import tracking

y_train = dataset.radio_map_tensor(train_df)
y_val = dataset.radio_map_tensor(val_df)
y_test = dataset.radio_map_tensor(test_df)
print(f"target tensor per sample: {y_train.shape[1:]}  (cell-bands x grid cells)")

with tracking.start_run(cfg, run_name="surrogate"):
    tracking.log_config(cfg)
    model = instantiate(cfg.surrogate)
    model.fit(x_train, y_train, x_val, y_val)

## 5. Radio-map error — layer 1

Prediction error in dB against the Sionna-RT reference, over every
`(cell_band, grid_cell)` entry.

Report error **near the KPI thresholds** separately. Hole rate is a hard cut at
−120 dBm and weak rate at −90 dBm, so a dB of error at −119 flips a
classification while the same dB at −70 changes nothing. A good global RMSE with
the error concentrated at the coverage edge is the failure mode to look for.

In [ ]:
from src.evaluation import metrics

pred = model.predict(x_test)
map_err = metrics.radio_map_error(y_test, pred, cfg)
pd.Series(map_err).to_frame("dB")

# TODO: break the error down by distance from the hole and weak thresholds, and
# by band — a surrogate accurate on the low band and poor on the high one will
# mispredict band coordination specifically, which is KPI 4.

## 6. Derived-KPI error — layer 2

**This is what acceptance is decided on.** Run `src/kpi/` over the predicted
maps and compare against the KPIs of the true maps — the same function, two
different inputs.

Never averaged across KPIs. A surrogate that is excellent on weak rate and
useless on hole rate is useless, because hole rate is the highest-priority
objective, and a mean over the five hides exactly that.

Report in each KPI's own units. The acceptance test is a comparison against the
improvement a result claims, and a dimensionless normalised error cannot be
compared against anything.

In [ ]:
from src.kpi import vector

kpi_cols = list(cfg.kpi.order)
k_true = np.array([list(vector.kpi_vector(m, rho, table, cfg).values()) for m in y_test])
k_pred = np.array([list(vector.kpi_vector(m, rho, table, cfg).values()) for m in pred])

report = metrics.prediction_error(k_true, k_pred, tuple(kpi_cols))
pd.DataFrame(report).T[["mae", "rmse", "bias", "rank_correlation"]]

## 7. Error near the optimum, and on held-out scenarios

**The numbers that actually matter.** An optimizer spends its time in the best
region of the space, so error averaged over the whole dataset is dominated by
configurations it would never propose. A surrogate can look accurate globally
while being useless exactly where it gets used.

Rank by the *true* objective, not the predicted one — ranking by prediction
selects the configurations the surrogate is most optimistic about and biases the
error downward in the very region being examined.

Then report the same error per held-out scenario. Generalising to a new tilt in
a scene the model has seen is a much weaker claim than generalising to a
perturbed environment (PROJECT.md section 12), and only the second one supports
anything said about simulation-to-reality robustness.

In [ ]:
true_objective = np.array(
    [vector.scalarize(dict(zip(kpi_cols, row, strict=True)), cfg) for row in k_true]
)
near = metrics.error_near_optimum(k_true, k_pred, true_objective, tuple(kpi_cols), quantile=0.1)
display(pd.DataFrame(near).T[["mae", "rmse", "bias"]])

# TODO: group by test_df["scenario_id"] and report per-scenario MAE. A large
# spread across scenarios is the sim-to-reality result, not a nuisance.

## 8. Ranking fidelity

Both optimizers use the surrogate to *choose between* configurations. A model
with a constant bias but the right ordering optimises perfectly; an unbiased one
that shuffles the ranking does not. Weight this more heavily than MAE when
deciding whether the surrogate is good enough.

Plotted on the derived KPIs, because that is what the optimizers compare.

In [ ]:
fig, axes = plt.subplots(1, len(kpi_cols), figsize=(16, 3.2))
for ax, (i, col) in zip(axes, enumerate(kpi_cols), strict=True):
    ax.scatter(k_true[:, i], k_pred[:, i], s=12, alpha=0.6)
    lims = [min(k_true[:, i].min(), k_pred[:, i].min()), max(k_true[:, i].max(), k_pred[:, i].max())]
    ax.plot(lims, lims, ls="--", lw=1)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("true")
axes[0].set_ylabel("predicted")
plt.tight_layout()

## 9. Acceptance decision, then freeze

Compare the radio-map error against `cfg.surrogate.acceptance.radio_map`, and
each derived-KPI error against `cfg.surrogate.acceptance.max_mae` — and against
the size of the improvement the optimization is expected to claim.

**If predicted hole rate is off by more than the improvement being claimed, the
result is noise.** That is the test. Record the decision here — a surrogate that
quietly ships below tolerance produces optimization results nobody can defend,
and the failure surfaces much later and far more expensively.

On acceptance the surrogate is **frozen** for the whole optimization phase
(PROJECT.md section 11.4). It is not retrained on candidates TuRBO or MARL
propose; doing so would put a Sionna-RT solve back inside the loop, which is the
expense the surrogate exists to avoid.

In [ ]:
acceptance = pd.DataFrame(
    {
        "mae": {k: report[k]["mae"] for k in kpi_cols},
        "max_mae": dict(cfg.surrogate.acceptance.max_mae),
    }
)
acceptance["passes"] = acceptance["mae"] <= acceptance["max_mae"]
display(acceptance)
print("radio-map error:", map_err)

# TODO: state the expected improvement per KPI, and check the MAE is smaller
# than it. Do not proceed to notebooks 05a/05b until this holds or the gap is
# recorded as a known limitation of every result that follows.

## 10. Persist

The transformer travels with the model. Saved apart, the reloaded surrogate
receives tilts on a different scale from the one it was trained on and
mispredicts silently.

The cell-band ordering travels too — a tilt vector cannot be interpreted without
the column order it was built with, and neither can a predicted map.

In [ ]:
model.transformer = transformer
model.save(cfg.surrogate.artifact_path)
print(f"wrote {cfg.surrogate.artifact_path}")

## 11. Handoff checklist

- [ ] The feature transform was fitted on the training partition only.
- [ ] The split held out whole **scenarios**, and the leak assertion in section 2
      passed.
- [ ] Radio-map error is reported in dB, including near the KPI thresholds.
- [ ] Derived-KPI errors are reported in each KPI's own units, never averaged.
- [ ] Near-optimum error and per-scenario error are both reported.
- [ ] The acceptance decision in section 9 is recorded, pass or fail.
- [ ] The artifact carries the transformer, the cell-band ordering and the grid
      geometry the map is indexed by.
- [ ] The surrogate is frozen. Notebooks 05a and 05b load it and do not refit it.

**Blocking gap.** `src.surrogate` still implements the KPI-predicting interface;
`dataset.radio_map_tensor` and `metrics.radio_map_error` do not exist yet.